# Ejercicio 3.3 · Reranking sobre el híbrido

La búsqueda sigue siendo la mezcla de 3.2: denso de `qwen3-embedding:0.6b` y BM25, con pesos 0.5/0.5, sobre los 2.698 fragmentos de Chroma. No se vuelve a embeber el corpus.

El reranking mira la pregunta y cada fragmento juntos, y vuelve a ordenar un grupo más grande. Se comparan tres versiones. v1 es el top 5 de la mezcla. v2 pide a Cohere `rerank-v3.5` que reordene los 30 primeros. v3 hace lo mismo con `bge-reranker-v2-m3` cuantizado, en local. Las preguntas están en español y los PDF en inglés, así que el reranker local es el multilingüe y no el `bge-reranker-base`, que solo se entrenó en inglés.

Cohere en trial no se cobra, pero hace falta una clave en `.env`. Sin ella, v2 queda escrito y no se llama. v3 y la cadena final corren igual.

## 1. Los mismos recuperadores

`k` de cada lado se cambia según la versión. Con 5, la mezcla solo puede reordenar lo que ya está arriba. Con 30, un fragmento que iba sexto puede entrar y el reranker decidir si sube.

In [1]:
import math
import os
import re
import time
from pathlib import Path

os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import chromadb
from fastembed.rerank.cross_encoder import TextCrossEncoder
from langchain_chroma import Chroma
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_core.retrievers import BaseRetriever
from pydantic import ConfigDict, Field
from rank_bm25 import BM25Okapi


class BM25Retriever(BaseRetriever):
    """BM25 local. La misma forma que el de langchain-community, sin ese paquete."""

    vectorizer: object = None
    docs: list[Document] = Field(default_factory=list, repr=False)
    k: int = 4
    preprocess_func: object = None
    model_config = ConfigDict(arbitrary_types_allowed=True)

    @classmethod
    def from_documents(cls, documents, *, preprocess_func, **kwargs):
        lista = list(documents)
        vectorizer = BM25Okapi([preprocess_func(doc.page_content) for doc in lista])
        return cls(vectorizer=vectorizer, docs=lista, preprocess_func=preprocess_func, **kwargs)

    def _get_relevant_documents(self, query: str, *, run_manager) -> list[Document]:
        return list(self.vectorizer.get_top_n(self.preprocess_func(query), self.docs, n=self.k))

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings

RAIZ = Path(".").resolve()
CLAVE_COHERE = ""
archivo_env = RAIZ / ".env"
if archivo_env.exists():
    for linea in archivo_env.read_text().splitlines():
        linea = linea.strip()
        if not linea or linea.startswith("#") or "=" not in linea:
            continue
        nombre, valor = linea.split("=", 1)
        os.environ.setdefault(nombre.strip(), valor.strip().strip('"').strip("'"))
CLAVE_COHERE = os.environ.get("COHERE_API_KEY", "").strip()

CHROMA_DIR = RAIZ / "chroma_db"
COLECCION = "python_pdf_500"
MODELO_EMBEDDING = "qwen3-embedding:0.6b"
MODELO_LLM = "llama3.2"
MODELO_BGE = "BAAI/bge-reranker-v2-m3-int8"
PREFIJO_QUERY = (
    "Instruct: Given a question about Python, retrieve the relevant documentation passage\n"
    "Query:"
)
C_RRF = 60
K_FINAL = 5
K_RETRIEVAL = 30
UMBRAL = 0.5


def tokenizar(texto: str) -> list[str]:
    return re.findall(r"[a-z0-9_]+", texto.lower())


cliente = chromadb.PersistentClient(path=str(CHROMA_DIR))
cruda = cliente.get_collection(COLECCION)
guardado = cruda.get(include=["documents", "metadatas"])
documentos = [
    Document(page_content=texto, metadata=meta, id=identificador)
    for identificador, texto, meta in zip(guardado["ids"], guardado["documents"], guardado["metadatas"])
]
print(f"{len(documentos)} fragmentos. Cohere: {'clave cargada' if CLAVE_COHERE else 'sin clave, v2 no se llama'}")


2698 fragmentos. Cohere: sin clave, v2 no se llama


## 2. Mezcla, Cohere y BGE

El denso cachea el vector de cada pregunta. BGE se carga una vez. Cohere, si hay clave, usa `POST /v2/rerank` con `rerank-v3.5`. Una búsqueda de Cohere admite hasta 100 documentos; nuestros fragmentos rondan los 500 caracteres, por debajo del corte de 500 tokens a partir del cual Cohere los parte y cobra de más.

In [2]:
class EmbeddingsLocales(Embeddings):
    def __init__(self, modelo: str) -> None:
        self._interno = OllamaEmbeddings(model=modelo, keep_alive=1800, client_kwargs={"timeout": 120})
        self.cache: dict[str, list[float]] = {}

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._interno.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        if text not in self.cache:
            self.cache[text] = self._interno.embed_query(f"{PREFIJO_QUERY} {text}")
        return self.cache[text]


def sigmoide(valor: float) -> float:
    if valor >= 0:
        z = math.exp(-valor)
        return 1.0 / (1.0 + z)
    z = math.exp(valor)
    return z / (1.0 + z)


embeddings = EmbeddingsLocales(MODELO_EMBEDDING)
almacen = Chroma(client=cliente, collection_name=COLECCION, embedding_function=embeddings)
denso = almacen.as_retriever(search_kwargs={"k": K_RETRIEVAL})
bm25 = BM25Retriever.from_documents(documentos, preprocess_func=tokenizar)
bm25.k = K_RETRIEVAL
ensemble = EnsembleRetriever(retrievers=[denso, bm25], weights=[0.5, 0.5], c=C_RRF, id_key="chunk_id")

inicio = time.perf_counter()
reranker_bge = TextCrossEncoder(
    model_name=MODELO_BGE, cache_dir=str(RAIZ / ".fastembed_cache"), cuda=False, threads=8
)
notas_bge: dict[tuple[str, str], float] = {}
print(f"BGE cargado en {time.perf_counter() - inicio:.1f} s")


def candidatos(pregunta: str, k: int) -> tuple[list[Document], float]:
    denso.search_kwargs["k"] = k
    bm25.k = k
    inicio = time.perf_counter()
    bruto = ensemble.invoke(pregunta)
    segundos = time.perf_counter() - inicio
    vistos: set[str] = set()
    unicos: list[Document] = []
    for doc in bruto:
        identificador = doc.metadata["chunk_id"]
        if identificador in vistos:
            continue
        vistos.add(identificador)
        unicos.append(doc)
        if len(unicos) == k:
            break
    return unicos, segundos


def rerank_bge(pregunta: str, docs: list[Document]) -> tuple[list[tuple[Document, float]], float]:
    faltan = [doc for doc in docs if (pregunta, doc.metadata["chunk_id"]) not in notas_bge]
    segundos = 0.0
    if faltan:
        inicio = time.perf_counter()
        notas = list(reranker_bge.rerank(pregunta, [doc.page_content for doc in faltan], batch_size=16))
        segundos = time.perf_counter() - inicio
        for doc, nota in zip(faltan, notas):
            notas_bge[(pregunta, doc.metadata["chunk_id"])] = float(nota)
    orden = sorted(docs, key=lambda doc: notas_bge[(pregunta, doc.metadata["chunk_id"])], reverse=True)
    return [(doc, notas_bge[(pregunta, doc.metadata["chunk_id"])]) for doc in orden], segundos


def rerank_cohere(pregunta: str, docs: list[Document], top_n: int) -> tuple[list[tuple[Document, float]], float]:
    import json
    from urllib.request import Request, urlopen

    cuerpo = json.dumps({
        "model": "rerank-v3.5",
        "query": pregunta,
        "documents": [doc.page_content for doc in docs],
        "top_n": top_n,
    }).encode()
    peticion = Request(
        "https://api.cohere.com/v2/rerank",
        data=cuerpo,
        headers={
            "Authorization": f"Bearer {CLAVE_COHERE}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    inicio = time.perf_counter()
    with urlopen(peticion, timeout=60) as respuesta:
        datos = json.load(respuesta)
    segundos = time.perf_counter() - inicio
    orden = [(docs[fila["index"]], float(fila["relevance_score"])) for fila in datos["results"]]
    return orden, segundos


def con_nota(doc: Document, nota: float, motor: str) -> Document:
    probabilidad = nota if motor == "cohere" else sigmoide(nota)
    meta = {**doc.metadata, "rerank": nota, "rerank_01": probabilidad, "motor": motor}
    return Document(page_content=doc.page_content, metadata=meta, id=doc.id)


def retrieve_and_rerank(pregunta: str, k_retrieval: int = K_RETRIEVAL, k_final: int = K_FINAL, motor: str = "bge"):
    pool, t_busqueda = candidatos(pregunta, k_retrieval)
    if motor == "cohere":
        orden, t_rerank = rerank_cohere(pregunta, pool, top_n=min(k_final, len(pool)) if k_final < len(pool) else len(pool))
    else:
        orden, t_rerank = rerank_bge(pregunta, pool)
    docs = [con_nota(doc, nota, motor) for doc, nota in orden[:k_final]]
    return docs, t_busqueda, t_rerank

BGE cargado en 1.3 s


## 3. Las quince preguntas

El fragmento relevante es el que contiene la frase ancla, la misma que se comprobó a mano en 3.2. En `id_nubmer`, `re.IGNORECASE` y `yield from` hay más de uno: todos cuentan. Precision@5 es la media de cuántos de los cinco devueltos son de ese conjunto. Con una sola ancla el máximo es 0.20, así que también anoto en cuántas preguntas el fragmento correcto entra en el top 5.

In [3]:
PREGUNTAS = [
    {"id": "goodbye", "tipo": "específica", "pregunta": "Goodbye, world!", "ancla": "Goodbye, world!"},
    {"id": "lispers", "tipo": "específica", "pregunta": "paper intended for lispers", "ancla": "lispers"},
    {"id": "recursion", "tipo": "específica", "pregunta": "sys.setrecursionlimit", "ancla": "setrecursionlimit"},
    {"id": "nubmer", "tipo": "específica", "pregunta": "id_nubmer", "ancla": "id_nubmer"},
    {"id": "ignorecase", "tipo": "específica", "pregunta": "re.IGNORECASE", "ancla": "re.IGNORECASE"},
    {
        "id": "propiedad",
        "tipo": "semántica",
        "pregunta": "por qué un objeto no puede cambiar el comportamiento de un atributo que la clase calcula al leerlo",
        "ancla": "instances cannot override the behavior",
    },
    {
        "id": "pickle",
        "tipo": "semántica",
        "pregunta": "por qué es peligroso recuperar un objeto guardado en disco si quien lo escribió no es de confianza",
        "ancla": "untrusted source can execute arbitrary code",
    },
    {
        "id": "anidadas",
        "tipo": "semántica",
        "pregunta": "qué ocurre con los nombres que una función de dentro usa pero que fueron creados en la función de fuera",
        "ancla": "Free variables used in the nested function",
    },
    {
        "id": "limpieza",
        "tipo": "semántica",
        "pregunta": "cómo fuerzas un bloque de limpieza que se ejecuta tanto si el intento sale bien como si salta un error",
        "ancla": "define clean-up actions that must be executed",
    },
    {
        "id": "corchetes",
        "tipo": "semántica",
        "pregunta": "una manera breve de construir una secuencia nueva a partir de otra, escrita entre corchetes",
        "ancla": "List comprehensions provide a concise way",
    },
    {
        "id": "hash",
        "tipo": "mixta",
        "pregunta": "qué tiene que cumplir __hash__ para que dos objetos iguales sirvan de clave en un conjunto",
        "ancla": "objects which compare equal have the same hash",
    },
    {
        "id": "generador",
        "tipo": "mixta",
        "pregunta": "una generator expression frente a la lista equivalente, cuál ahorra memoria",
        "ancla": "parentheses instead of brackets",
    },
    {
        "id": "closed",
        "tipo": "mixta",
        "pregunta": "en el ejemplo de with open, qué valor tiene f.closed al salir del bloque",
        "ancla": "f.closed",
    },
    {
        "id": "dylan",
        "tipo": "mixta",
        "pregunta": "el method resolution order que Python 2.3 copió de Dylan",
        "ancla": "people working on Dylan",
    },
    {
        "id": "yieldfrom",
        "tipo": "mixta",
        "pregunta": "diferencia de sintaxis entre yield from y un yield de una sola expresión",
        "ancla": "yield from <expr>",
    },
]

verdad = {
    pregunta["id"]: {doc.metadata["chunk_id"] for doc in documentos if pregunta["ancla"] in doc.page_content}
    for pregunta in PREGUNTAS
}
for pregunta in PREGUNTAS:
    print(f"{pregunta['id']:<12} {pregunta['tipo']:<12} relevantes: {len(verdad[pregunta['id']])}")


def precision(identificadores: list[str], buenos: set[str]) -> float:
    return sum(identificador in buenos for identificador in identificadores[:K_FINAL]) / K_FINAL


def ids_de(docs: list[Document]) -> list[str]:
    return [doc.metadata["chunk_id"] for doc in docs]

goodbye      específica   relevantes: 1
lispers      específica   relevantes: 1
recursion    específica   relevantes: 1
nubmer       específica   relevantes: 2
ignorecase   específica   relevantes: 3
propiedad    semántica    relevantes: 1
pickle       semántica    relevantes: 1
anidadas     semántica    relevantes: 1
limpieza     semántica    relevantes: 1
corchetes    semántica    relevantes: 1
hash         mixta        relevantes: 1
generador    mixta        relevantes: 1
closed       mixta        relevantes: 1
dylan        mixta        relevantes: 1
yieldfrom    mixta        relevantes: 4


## 4. v1, v2 y v3

Antes de cronometrar se embebe cada pregunta una vez y se hace una pasada corta de BGE, para que la tabla no mezcle la carga del modelo con la búsqueda. v1 pide 5 a cada lado y se queda el top 5 de la fusión. v3 pide 30, los reordena y se queda 5. El tiempo de v3 es la búsqueda de esos 30 más el reranker.

In [4]:
# Calienta embeddings y una pasada del cross-encoder. No entra en los tiempos.
for pregunta in PREGUNTAS:
    embeddings.embed_query(pregunta["pregunta"])
_ = rerank_bge("yield", documentos[:2])

filas = []
for pregunta in PREGUNTAS:
    buenos = verdad[pregunta["id"]]
    v1_docs, t_v1 = candidatos(pregunta["pregunta"], K_FINAL)
    v3_docs, t_busqueda, t_bge = retrieve_and_rerank(pregunta["pregunta"], K_RETRIEVAL, K_FINAL, "bge")
    fila = {
        "id": pregunta["id"],
        "tipo": pregunta["tipo"],
        "v1": ids_de(v1_docs),
        "v3": ids_de(v3_docs),
        "p_v1": precision(ids_de(v1_docs), buenos),
        "p_v3": precision(ids_de(v3_docs), buenos),
        "t_v1": t_v1,
        "t_busqueda30": t_busqueda,
        "t_bge": t_bge,
        "v3_docs": v3_docs,
        "v1_docs": v1_docs,
    }
    if CLAVE_COHERE:
        v2_docs, t_busqueda_v2, t_cohere = retrieve_and_rerank(pregunta["pregunta"], K_RETRIEVAL, K_FINAL, "cohere")
        fila["v2"] = ids_de(v2_docs)
        fila["p_v2"] = precision(fila["v2"], buenos)
        fila["t_cohere"] = t_cohere
        fila["t_v2"] = t_busqueda_v2 + t_cohere
    filas.append(fila)


def media(valores: list[float]) -> float:
    return sum(valores) / len(valores)


def aciertos(clave: str) -> int:
    return sum(fila[clave] > 0 for fila in filas)


print(f"{'id':<12} {'tipo':<12} {'P@5 v1':>8} {'P@5 bge':>8}  top5 v1 y bge, * = relevante")
for fila in filas:
    buenos = verdad[fila["id"]]
    def marca(identificadores: list[str]) -> str:
        return " ".join(("*" if identificador in buenos else "") + identificador[:6] for identificador in identificadores)

    extra = ""
    if "p_v2" in fila:
        extra = f"  cohere {fila['p_v2']:.2f}"
    print(f"{fila['id']:<12} {fila['tipo']:<12} {fila['p_v1']:8.2f} {fila['p_v3']:8.2f}{extra}")
    print(f"{'':<12} {'v1':<12} {marca(fila['v1'])}")
    print(f"{'':<12} {'bge':<12} {marca(fila['v3'])}")
    if "v2" in fila:
        print(f"{'':<12} {'cohere':<12} {marca(fila['v2'])}")

print()
print(
    f"Precision@5 media   v1 {media([f['p_v1'] for f in filas]):.3f}"
    f"   bge {media([f['p_v3'] for f in filas]):.3f}"
    + (f"   cohere {media([f['p_v2'] for f in filas]):.3f}" if CLAVE_COHERE else "")
)
print(
    f"Preguntas con el fragmento correcto en el top 5:"
    f"  v1 {aciertos('p_v1')}/15   bge {aciertos('p_v3')}/15"
    + (f"   cohere {aciertos('p_v2')}/15" if CLAVE_COHERE else "")
)
print(
    f"Latencia media   v1 {1000 * media([f['t_v1'] for f in filas]):.0f} ms"
    f"   búsqueda de 30 {1000 * media([f['t_busqueda30'] for f in filas]):.0f} ms"
    f"   BGE {1000 * media([f['t_bge'] for f in filas]):.0f} ms"
    f"   v3 total {1000 * media([f['t_busqueda30'] + f['t_bge'] for f in filas]):.0f} ms"
)
if CLAVE_COHERE:
    print(f"Cohere rerank {1000 * media([f['t_cohere'] for f in filas]):.0f} ms   v2 total {1000 * media([f['t_v2'] for f in filas]):.0f} ms")
else:
    print("Cohere no se ha llamado. La clave no está en .env.")

print()
print("Preguntas en las que BGE cambia el conjunto del top 5")
for fila in filas:
    if set(fila["v1"]) == set(fila["v3"]):
        continue
    buenos = verdad[fila["id"]]
    entran = [doc for doc in fila["v3_docs"] if doc.metadata["chunk_id"] not in set(fila["v1"])]
    print(f"\n{fila['id']}  P@5 {fila['p_v1']:.2f} -> {fila['p_v3']:.2f}")
    for doc in entran:
        relevante = "relevante" if doc.metadata["chunk_id"] in buenos else "no relevante"
        texto = doc.page_content[:180].replace("\n", " ")
        print(f"  entra {doc.metadata['chunk_id'][:8]}  {relevante}  nota {doc.metadata['rerank_01']:.2f}  {texto}")

id           tipo           P@5 v1  P@5 bge  top5 v1 y bge, * = relevante
goodbye      específica       0.20     0.20
             v1           426200 *20eacc 512bbf 210f04 47ddb1
             bge          *20eacc a7a159 68fd3a 210f04 9cefeb
lispers      específica       0.20     0.20
             v1           5c8cab *96d268 4b5665 b81dd6 aae14a
             bge          *96d268 b81dd6 142b6f cacd9d 5e0505
recursion    específica       0.20     0.20
             v1           *512bbf cdc53d 1876e8 5bceb1 da196c
             bge          *512bbf 97722a cdc53d a891ba 5a73cd
nubmer       específica       0.40     0.40
             v1           861d24 *aac71f 735f50 *88d763 994bca
             bge          *aac71f *88d763 8e24f4 e0f587 643969
ignorecase   específica       0.60     0.60
             v1           *2ae00c *b574e1 *361355 657226 a70793
             bge          *2ae00c *361355 *b574e1 a70793 313094
propiedad    semántica        0.20     0.20
             v1           *151106 ee

## 5. Bonus 1 · Cuántos candidatos

Se recupera con k = 10, 30, 50 y 100, BGE reordena ese grupo y se miden los mismos cinco. Un fragmento que ya se puntuó no se vuelve a pasar por el modelo: la latencia de reordenar 30 está en la tabla anterior.

In [5]:

barrido = {}
for k in (10, 30, 50, 100):
    quien = []
    precisiones = []
    for pregunta in PREGUNTAS:
        docs, _, _ = retrieve_and_rerank(pregunta["pregunta"], k, K_FINAL, "bge")
        p = precision(ids_de(docs), verdad[pregunta["id"]])
        precisiones.append(p)
        if p > 0:
            quien.append(pregunta["id"])
    barrido[k] = {"precision": media(precisiones), "quien": quien}

print(f"{'k':>6} {'P@5':>8} {'con acierto':>12}")
base = set(barrido[30]["quien"])
for k, fila in barrido.items():
    print(f"{k:6d} {fila['precision']:8.3f} {len(fila['quien']):10d}/15")
    faltan = sorted(base - set(fila["quien"]))
    nuevas = sorted(set(fila["quien"]) - base)
    if faltan:
        print(f"       salen respecto a k=30: {', '.join(faltan)}")
    if nuevas:
        print(f"       entran respecto a k=30: {', '.join(nuevas)}")
ms_por_fragmento = 1000 * media([f["t_bge"] for f in filas]) / K_RETRIEVAL
print(f"BGE, medido al reordenar 30: {ms_por_fragmento:.0f} ms por fragmento")


     k      P@5  con acierto
    10    0.240         14/15
    30    0.240         14/15
    50    0.240         14/15
   100    0.213         12/15
       salen respecto a k=30: corchetes, pickle
BGE, medido al reordenar 30: 90 ms por fragmento



## 6. Cadena final y umbral

La cadena sustituye la búsqueda por `retrieve_and_rerank`: trae 30, BGE los ordena y el modelo ve esos cinco. Las cinco pruebas comparan ese contexto con el top 5 híbrido, sin umbral, porque si no, el corte se come la diferencia.

El umbral va aparte. Pasan los fragmentos con score mayor que 0.5 y, si no llegan a dos, la cadena no llama al modelo. En Cohere ese 0.5 es un `relevance_score` ya calibrado. BGE devuelve un logit y aquí se le aplica el sigmoide, así que 0.5 significa logit 0. Abajo se ve la nota del fragmento correcto: si no llega a 0.5, el corte descarta una pregunta que el reranker sí había colocado.


In [6]:

def formatear_notas(docs: list[Document]) -> str:
    return " ".join(f"{doc.metadata['rerank_01']:.2f}" for doc in docs)


def documentos_para_llm(pregunta: str, motor: str = "bge") -> list[Document] | None:
    ordenados, _, _ = retrieve_and_rerank(pregunta, K_RETRIEVAL, K_RETRIEVAL, motor)
    pasan = [doc for doc in ordenados if doc.metadata["rerank_01"] > UMBRAL]
    if len(pasan) < 2:
        return None
    return pasan[:K_FINAL]


def formatear(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_template(
    "Responde a la pregunta usando solo el contexto. "
    "Si el contexto no contiene la respuesta, di que no lo sabes. No inventes.\n\n"
    "Contexto:\n{context}\n\n"
    "Pregunta: {question}"
)
llm = ChatOllama(model=MODELO_LLM, temperature=0, num_ctx=8192, num_predict=180, keep_alive=1800)
generar = prompt | llm | StrOutputParser()


def responder(pregunta: str) -> str:
    docs = documentos_para_llm(pregunta)
    if docs is None:
        return "No contestable: menos de dos fragmentos superan 0.5."
    return generar.invoke({"context": formatear(docs), "question": pregunta}).strip()


cadena = RunnableLambda(lambda pregunta: responder(pregunta))

print("Nota del fragmento correcto dentro de los 30, y si el umbral deja pasar la pregunta")
for pregunta in PREGUNTAS:
    ordenados, _, _ = retrieve_and_rerank(pregunta["pregunta"], K_RETRIEVAL, K_RETRIEVAL, "bge")
    buenos = verdad[pregunta["id"]]
    notas = [doc.metadata["rerank_01"] for doc in ordenados if doc.metadata["chunk_id"] in buenos]
    docs = documentos_para_llm(pregunta["pregunta"])
    nota = f"{max(notas):.3f}" if notas else "fuera de los 30"
    estado = "pasa" if docs is not None else "no contestable"
    print(f"{pregunta['id']:<12} mejor nota {nota:<14} {estado}")

fuera = "cómo uno dos DataFrames de pandas con merge"
print()
print("Fuera de corpus:", fuera)
print(cadena.invoke(fuera))

print()
for nombre in ("corchetes", "anidadas", "hash", "pickle", "goodbye"):
    fila = next(item for item in filas if item["id"] == nombre)
    pregunta = next(item for item in PREGUNTAS if item["id"] == nombre)
    print("=" * 70)
    print(nombre, pregunta["pregunta"])
    print(f"P@5 híbrido {fila['p_v1']:.2f}   P@5 bge {fila['p_v3']:.2f}")
    t0 = time.perf_counter()
    solo_hibrido = generar.invoke({"context": formatear(fila["v1_docs"]), "question": pregunta["pregunta"]}).strip()
    t_h = time.perf_counter() - t0
    t0 = time.perf_counter()
    con_rerank = generar.invoke({"context": formatear(fila["v3_docs"]), "question": pregunta["pregunta"]}).strip()
    t_r = time.perf_counter() - t0
    print(f"-- solo híbrido ({t_h:.1f} s)")
    print(solo_hibrido)
    print(f"-- híbrido + rerank ({t_r:.1f} s)")
    print(con_rerank)
    print("-- con umbral 0.5")
    print(cadena.invoke(pregunta["pregunta"]))


Nota del fragmento correcto dentro de los 30, y si el umbral deja pasar la pregunta
goodbye      mejor nota 0.848          no contestable
lispers      mejor nota 0.893          no contestable
recursion    mejor nota 0.153          no contestable
nubmer       mejor nota 0.844          pasa
ignorecase   mejor nota 0.742          pasa
propiedad    mejor nota 0.020          no contestable
pickle       mejor nota 0.000          no contestable
anidadas     mejor nota fuera de los 30 no contestable
limpieza     mejor nota 0.005          no contestable
corchetes    mejor nota 0.005          no contestable
hash         mejor nota 0.454          no contestable
generador    mejor nota 0.026          no contestable
closed       mejor nota 0.403          no contestable
dylan        mejor nota 0.939          pasa
yieldfrom    mejor nota 0.055          no contestable

Fuera de corpus: cómo uno dos DataFrames de pandas con merge
No contestable: menos de dos fragmentos superan 0.5.

corchetes una maner

## 7. Bonus 3 · Qué pesa en la factura

Cifras de la lista de precios que consume la API, no del ensayo. `rerank-v3.5` sale a 2 USD por cada 1.000 búsquedas. Cohere cuenta como una búsqueda una pregunta con hasta 100 documentos, y parte un documento solo si pregunta más texto pasan de 500 tokens. `embed-v4.0` sale a 0,12 USD por millón de tokens. `command-r-08-2024` sale a 0,15 USD por millón de entrada y 0,60 USD por millón de salida, la misma banda que gpt-4o-mini.

El pipeline de este portátil no paga nada de eso: los vectores y `llama3.2` son locales, y BGE también. La cuenta de abajo es la misma carga, 10.000 preguntas al mes, si cada etapa fuera de Cohere.

In [7]:
consultas = 10_000
caracteres = [len(doc.page_content) for doc in documentos]
tokens_fragmento = max(caracteres) / 4
print(f"Fragmento más largo: {max(caracteres)} caracteres, unos {tokens_fragmento:.0f} tokens. Cabe en una búsqueda.")

usd_rerank = consultas * 0.002
tokens_pregunta = 40
usd_embed_preguntas = consultas * tokens_pregunta * 0.12 / 1_000_000
tokens_corpus = sum(caracteres) / 4
usd_embed_corpus = tokens_corpus * 0.12 / 1_000_000
tokens_entrada = 5 * (sum(caracteres) / len(caracteres) / 4) + 200
tokens_salida = 150
usd_llm = consultas * (tokens_entrada * 0.15 + tokens_salida * 0.60) / 1_000_000

print(f"Rerank 10.000 búsquedas          {usd_rerank:6.2f} USD/mes")
print(f"Embeddings de las preguntas       {usd_embed_preguntas:6.2f} USD/mes")
print(f"Embeddings del corpus, una vez    {usd_embed_corpus:6.2f} USD")
print(f"LLM, unas {tokens_entrada:.0f} tokens de entrada y {tokens_salida} de salida   {usd_llm:6.2f} USD/mes")
print(f"Total mensual en la nube          {usd_rerank + usd_embed_preguntas + usd_llm:6.2f} USD")
print("En local, las tres etapas salen a 0 USD. La trial key de Cohere tampoco cobra, y no sirve para estas 10.000 al mes.")

Fragmento más largo: 500 caracteres, unos 125 tokens. Cabe en una búsqueda.
Rerank 10.000 búsquedas           20.00 USD/mes
Embeddings de las preguntas         0.05 USD/mes
Embeddings del corpus, una vez      0.03 USD
LLM, unas 735 tokens de entrada y 150 de salida     2.00 USD/mes
Total mensual en la nube           22.05 USD
En local, las tres etapas salen a 0 USD. La trial key de Cohere tampoco cobra, y no sirve para estas 10.000 al mes.



## 8. Lectura

Precision@5 media: híbrido 0.227, BGE 0.240. El fragmento correcto entra en 13 de 15 preguntas sin reranking y en 14 con él. La única que cambia de verdad es la de las list comprehensions: la mezcla no la traía y BGE la mete en el top 5. `anidadas` sigue fuera incluso de los 30, así que reordenar no puede inventarla.

Eso no se traduce en respuestas mejores de manera pareja.

- **Mejora.** Con el fragmento de las list comprehensions en el contexto, `llama3.2` explica la comprensión de listas. Con el top 5 híbrido decía que no lo sabía.
- **Empeora.** En `__hash__` y en pickle el fragmento correcto ya estaba. La mezcla respondía bien (mismo hash si comparan igual; código arbitrario si la fuente no es de fiar). Tras reordenar, el modelo dice que el contexto no trae la respuesta, aunque el fragmento sigue entre los cinco. En pickle la nota de ese fragmento es 0.00: queda en el top 5 porque los demás están igual de bajos, y el modelo no lo usa.
- **Deja de inventar.** En las funciones anidadas ninguno de los dos trae el párrafo de las free variables. La mezcla se inventa una explicación con `nonlocal`. El contexto reordenado no da pie y el modelo dice que no lo sabe.

**Latencia.** La mezcla tarda 9 ms. Reordenar 30 fragmentos con BGE tarda 2,8 s, unos 95 ms por fragmento. En este corpus el reranker es casi todo el tiempo de la consulta.

**k.** Con 10, 30 y 50 el resultado es el mismo, 14 de 15. Con 100 se caen las list comprehensions y pickle, las dos que BGE puntúa cerca de cero y que solo aguantan si el grupo no se llena de ruido. El rendimiento decreciente aparece antes de 30, y pasar de 50 empeora.

**Umbral.** 0.5 en el sigmoide de BGE no es el 0.5 de Cohere. El fragmento de `Goodbye, world!` saca 0.85 y el de `lispers` 0.89, pero son el único fragmento por encima del corte, y la regla de «menos de dos» los declara no contestables. Las paráfrasis ni llegan: las list comprehensions sacan 0.005 y el corte se lleva la única mejora. La pregunta de pandas, que no está en el corpus, sí queda cortada. El umbral sirve para lo de fuera. Con este modelo, en lo de dentro, borra aciertos.

**Coste, si las 10.000 consultas al mes fueran de pago.** `rerank-v3.5` sale a 2 USD por cada 1.000 búsquedas: 20 USD. Cada consulta es una sola búsqueda, porque pedimos menos de 100 fragmentos y el más largo tiene 500 caracteres, unos 125 tokens, por debajo del corte de 500. Embedir las preguntas con `embed-v4.0` son 0,05 USD. El corpus, una vez, 0,03 USD. Un LLM de la banda de `command-r-08-2024` o gpt-4o-mini, con cinco fragmentos en el prompt, son unos 2 USD. El rerank pesa unas diez veces más que el modelo que responde y unas cuatrocientas más que los embeddings de las preguntas. Aquí las tres etapas van en local y salen a 0. Cohere en trial tampoco cobra, pero no da para 10.000 al mes y en esta máquina no hay `COHERE_API_KEY`, así que v2 no se ha ejecutado. La llamada está en el notebook: con la clave en `.env` entra en la misma tabla.
